# Reporters vs essentiality

For each geneKO, count the number of reporters whose mAP distinctiveness exceeds 0.1, bin genes by CERES essentiality score (non-essential / moderate / essential), and compare the distributions with a violin plot. Kruskal–Wallis tests for any difference across the three bins; pairwise Mann–Whitney U brackets show which pairs differ.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["svg.fonttype"] = "none"

FIGURES_DIR = Path("../../output/figure_3")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV paths

Two inputs from `../../data/figures/figure_3/` (see README): the per-gene × per-reporter livecell-only distinctiveness matrix, and the CERES gene-effect table for the twist1k pool.

In [ ]:
FIGURE_DATA = Path("../../data/figures/figure_3")

MAP_CSV         = FIGURE_DATA / "gene_reporter_distinctiveness_livecell.csv"
GENE_EFFECT_CSV = FIGURE_DATA / "twist1k_pool_CERES.csv"

MAP_THRESHOLD = 0.1

## Load mAP matrix

Genes × reporters distinctiveness; drop the `all_combined` reference column so it isn't counted as a reporter.

In [ ]:
raw_df = pd.read_csv(MAP_CSV, index_col=0)
raw_df.index.name = "gene"
reporter_cols = [c for c in raw_df.columns if c != "all_combined"]
map_vals = raw_df[reporter_cols]
print(f"genes: {len(map_vals)}, reporters: {map_vals.shape[1]}")

## Load and unify CERES gene effect

The CSV has two gene-name columns (`dep_map_gene_name` and `Gene name`). Use `dep_map_gene_name` as the primary key; fall back to `Gene name` for any genes missing from the primary. Within each name column, average `gene_effect` across rows that map to the same gene.

In [ ]:
ge_df = pd.read_csv(GENE_EFFECT_CSV)
if "gene_effect" not in ge_df.columns:
    raise ValueError(f"'gene_effect' column not in {GENE_EFFECT_CSV}")

name_cols = [c for c in ["dep_map_gene_name", "Gene name"] if c in ge_df.columns]
if not name_cols:
    raise ValueError(f"No gene name columns found in {GENE_EFFECT_CSV}. Available: {list(ge_df.columns)}")

gene_effect_parts = [
    ge_df[[col, "gene_effect"]]
        .dropna(subset=[col, "gene_effect"])
        .groupby(col)["gene_effect"]
        .mean()
    for col in name_cols
]
gene_effect = gene_effect_parts[0].copy()
for fallback in gene_effect_parts[1:]:
    missing = ~fallback.index.isin(gene_effect.index)
    gene_effect = pd.concat([gene_effect, fallback[missing]])
print(f"gene_effect entries: {len(gene_effect)}")

## Join → genes present in both tables

In [ ]:
common_genes = sorted(set(map_vals.index) & set(gene_effect.index))
plot_gene_effect = gene_effect.loc[common_genes]
plot_map_vals = map_vals.loc[common_genes]

unmatched = sorted(set(map_vals.index) - set(gene_effect.index))
print(f"mAP matrix genes: {len(map_vals)}, matched: {len(common_genes)}, unmatched: {len(unmatched)}")
if unmatched:
    print(f"  unmatched: {unmatched}")

## Violin plot — # markers by essentiality bin

Bin genes by CERES essentiality score and plot the distribution of `# markers with mAP > MAP_THRESHOLD` per bin. Kruskal–Wallis tests for any difference across all three bins; pairwise Mann–Whitney U brackets show which pairs differ.

In [ ]:
BINS = [
    ("non-essential\n(0.5 to -0.5)",  -0.5,  0.5),
    ("moderate\n(-0.5 to -1.5)",      -1.5, -0.5),
    ("essential\n(-1.5 to -2.5)",     -2.5, -1.5),
]
PAIRS = [(0, 1), (1, 2), (0, 2)]

def stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

nm = (plot_map_vals > MAP_THRESHOLD).sum(axis=1)

bin_data, bin_labels = [], []
for label, lo, hi in BINS:
    mask = (plot_gene_effect >= lo) & (plot_gene_effect < hi)
    bin_data.append(nm[mask].values)
    bin_labels.append(f"{label}\n(n={mask.sum()})")

kw_stat, kw_p = stats.kruskal(*[d for d in bin_data if len(d) > 0])
mw_results = {}
for i, j in PAIRS:
    if len(bin_data[i]) > 0 and len(bin_data[j]) > 0:
        _, p = stats.mannwhitneyu(bin_data[i], bin_data[j], alternative="two-sided")
        mw_results[(i, j)] = p

fig, ax = plt.subplots(figsize=(7, 5))
parts = ax.violinplot(bin_data, positions=range(len(BINS)), showmedians=True, showextrema=True)
for pc in parts["bodies"]:
    pc.set_facecolor("steelblue")
    pc.set_alpha(0.6)
ax.set_xticks(range(len(BINS)))
ax.set_xticklabels(bin_labels, fontsize=10)
ax.set_ylabel(f"# markers with mAP > {MAP_THRESHOLD}", fontsize=12)
ax.set_title(f"Markers by essentiality bin  (mAP > {MAP_THRESHOLD})\nKruskal–Wallis p={kw_p:.2e}", fontsize=12)
ax.grid(True, axis="y", linestyle="--", alpha=0.35)

y_max = max((d.max() if len(d) else 0) for d in bin_data)
bracket_heights = [y_max * 1.05, y_max * 1.13, y_max * 1.21]
for k, (i, j) in enumerate(PAIRS):
    p = mw_results.get((i, j))
    if p is None:
        continue
    h = bracket_heights[k]
    ax.plot([i, i, j, j], [h * 0.97, h, h, h * 0.97], lw=1.2, color="black")
    ax.text((i + j) / 2, h * 1.005, f"{stars(p)} p={p:.2e}", ha="center", va="bottom", fontsize=8)

ax.set_ylim(top=y_max * 1.35)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"essentiality_bins_violin_thresh{MAP_THRESHOLD}.svg", bbox_inches="tight")
plt.show()